In [1]:
import sys
print(sys.executable)

C:\Users\vaish\anaconda3\envs\torch_env\python.exe


In [2]:
%pip install langchain langchain-community pypdf sentence-transformers rank_bm25 chromadb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [4]:
%pip install pymupdf


Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install langchain-community pymupdf langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [6]:
import time
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Path to your 600-page book
pdf_name = "data.pdf" 

print("🚀 Starting Optimized CPU Ingestion...")
start_time = time.time()

# 2. Use PyMuPDFLoader (Better for large books)
loader = PyMuPDFLoader(pdf_name)
pages = loader.load()

# 3. Chunking with smaller steps to save memory
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, # Smaller chunks are easier for CPUs
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pages)

end_time = time.time()

print(f"✅ SUCCESS!")
print(f"📄 Total Pages: {len(pages)}")
print(f"🧱 Total Chunks: {len(chunks)}")
print(f"⏱️ Time Taken: {end_time - start_time:.2f} seconds")

C:\Users\vaish\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Starting Optimized CPU Ingestion...
✅ SUCCESS!
📄 Total Pages: 607
🧱 Total Chunks: 3200
⏱️ Time Taken: 6.13 seconds


In [7]:
%pip install langchain-huggingface
%pip install sentence-transformers
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import time
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("🧠 Initializing Vector Store (Dense Search)...")
start_time = time.time()

# We use a small, fast embedding model optimized for CPUs
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the search database in your local memory
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    collection_name="research_collection"
)

end_time = time.time()
print(f"✅ Vector Store Created in {end_time - start_time:.2f} seconds")

🧠 Initializing Vector Store (Dense Search)...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2773.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector Store Created in 416.58 seconds


In [9]:
%pip install rank_bm25

In [10]:
import time
from langchain_community.retrievers import BM25Retriever

print("🔍 Initializing Sparse Search (BM25 Keyword Matching)...")
start_time = time.time()

# 1. Create the BM25 retriever from the existing chunks
bm25_retriever = BM25Retriever.from_documents(chunks)

# 2. Set it to find the top 5 exact matches
bm25_retriever.k = 5 

end_time = time.time()
print(f"✅ Sparse Index Created in {end_time - start_time:.2f} seconds")

🔍 Initializing Sparse Search (BM25 Keyword Matching)...
✅ Sparse Index Created in 1.49 seconds


In [11]:
%pip install langchain
%pip install -U langchain langchain-core langchain-community

Note: you may need to restart the kernel to use updated packages.
  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langgraph-1.1.6-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_prebuilt-1.0.9-py3-none-any.whl.metadata (5.2 kB)
Using cached langchain-1.2.15-py3-none-any.whl (112 kB)
Using cached langgraph-1.1.6-py3-none-any.whl (169 kB)
Using cached langgraph_prebuilt-1.0.9-py3-none-any.whl (35 kB)

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.2.23

   ---------------------------------------- 0/4 [langchain-core]
    Uninstalling langchain-core-1.2.23:
   ---------------------------------------- 0/4 [langchain-core]
      Successfully uninstalled langchain-core-1.2.23
   ---------------------------------------- 0/4 [langchain-core]
   ---------------------------------------- 0/4 [langchain-core]
   ---------------------------------------- 0/4 [langchain-core]
   ----------------------------------

In [15]:
import time

print("🔗 Manual Hybrid Fusion Starting...")
start_time = time.time()

def hybrid_search(query, k=):
    # 1. Semantic Search (The 161.65s Dense Brain)
    # This finds 'Meaning'
    dense_results = vector_db.similarity_search(query, k=k)
    
    # 2. Keyword Search (The 0.60s Sparse Index)
    # This finds 'Exact Words'
    sparse_results = bm25_retriever.invoke(query)[:k]
    
    # 3. Combine and Deduplicate
    # We merge both lists and ensure we don't have the same page twice
    all_results = dense_results + sparse_results
    unique_results = []
    seen_content = set()
    
    for doc in all_results:
        if doc.page_content not in seen_content:
            unique_results.append(doc)
            seen_content.add(doc.page_content)
            
    return unique_results[:k]

end_time = time.time()
print(f"🚀 MANUAL HYBRID SYSTEM LIVE! (Latency: {end_time - start_time:.4f}s)")

# --- QUICK TEST ---
test_query = "What is the definition of a project?"
results = hybrid_search(test_query)
print(f"✅ Success! Found {len(results)} relevant chunks from the textbook.")

🔗 Manual Hybrid Fusion Starting...
🚀 MANUAL HYBRID SYSTEM LIVE! (Latency: 0.0020s)
✅ Success! Found 40 relevant chunks from the textbook.


In [13]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [19]:
def reranked_hybrid_search(query, top_n=40, final_k=5):
    # A. Initial Retrieval (Hybrid Fusion)
    initial_chunks = hybrid_search(query, k=top_n)
    
    # B. Prepare pairs for the Reranker
    pairs = [[query, doc.page_content] for doc in initial_chunks]
    
    # C. Scoring
    start_rerank = time.time()
    scores = reranker.predict(pairs)
    end_rerank = time.time()
    
    # D. Attach scores to the metadata
    for i, doc in enumerate(initial_chunks):
        doc.metadata['rerank_score'] = float(scores[i])
    
    # E. NEW: Show scores for ALL chunks before picking the top ones
    print("\n--- 📊 RERANKER SCORECARD ---")
    print(f"{'Rank':<5} | {'Score':<10} | {'Content Snippet'}")
    print("-" * 70)
    
    # Sort them internally just for the display
    all_scored = sorted(initial_chunks, key=lambda x: x.metadata['rerank_score'], reverse=True)
    
    for rank, doc in enumerate(all_scored, 1):
        score = doc.metadata['rerank_score']
        # Clean up text for the display
        snippet = doc.page_content.replace('\n', ' ')[:60]
        print(f"#{rank:<4} | {score:>10.4f} | {snippet}...")
    print("-" * 70)
    
    print(f"🎯 Reranking Finished in {end_rerank - start_rerank:.4f}s")
    return all_scored[:final_k]

# --- TEST IT ---
test_query = "What is the definition of a project?"
best_chunks = reranked_hybrid_search(test_query, top_n=10)


--- 📊 RERANKER SCORECARD ---
Rank  | Score      | Content Snippet
----------------------------------------------------------------------
#1    |     2.9917 | projects. Project management has emerged because the charact...
#2    |     1.8148 | spread to the nonproﬁ t sector. Weddings, scout-o-ramas, fun...
#3    |     0.5000 | which are the importance of an organized plan for reaching a...
#4    |     0.1530 | well as a way to gain valuable experience within the   organ...
#5    |    -0.5381 | varied organizations continues to grow. Businesses regularly...
#6    |    -1.3142 | 22   CHAPTER 1 / PROJECTS IN CONTEMPORARY ORGANIZATIONS This...
#7    |    -1.3299 | dependencies, some or all unique elements, limited  resource...
#8    |    -1.8603 | keeps projects from competing with each other on inappropria...
#9    |    -3.7881 | another who suggested that we cut back on the “touchy-feely ...
#10   |    -3.9167 | team members. In Chapter 4 we will discuss some issues invol...
------------

In [20]:
# Test with the query that previously gave a negative score
vague_query = "What are the three primary goals of a project according to Meredith and Mantel?"

# Using the function we built to show ALL 10 rankings
vague_results = reranked_hybrid_search(vague_query, top_n=10)


--- 📊 RERANKER SCORECARD ---
Rank  | Score      | Content Snippet
----------------------------------------------------------------------
#1    |    -1.5837 | and desires along with the “speciﬁ ed” performance, as state...
#2    |    -3.3495 | project and are thus direct goals. Ancillary goals include i...
#3    |    -4.3411 | 4   CHAPTER 1 / PROJECTS IN CONTEMPORARY ORGANIZATIONS shown...
#4    |    -5.1128 | for the project’s existence, it is much easier to enlist and...
#5    |    -5.9779 | ment is the project’s environment, that is, those things or ...
#6    |    -6.9250 | keeps projects from competing with each other on inappropria...
#7    |    -7.1815 | associated with a project (e.g., antipollution). Whether cle...
#8    |    -7.2459 | scope of the project, and describe how the project’s desired...
#9    |    -8.5820 | mation?” As is true of so many things, tradition is probably...
#10   |   -10.2409 | P A R T I PROJECT INITIATION 35 As noted earlier, the materi...
------------

In [18]:
%pip install ollama

In [22]:
def ask_my_textbook_v2(query):
    # 1. HIGH-PRECISION RETRIEVAL (Phase 2.5)
    # We pull 40 candidates, judge them, and keep the best 5
    print(f"🔍 Deep Searching (Recall: 40, Precision: 5)...")
    context_chunks = reranked_hybrid_search(query, top_n=40, final_k=5)
    
    # Check the top score for our 'Kill Switch'
    top_score = context_chunks[0].metadata.get('rerank_score', 0)
    
    # 2. THE TRUTH FILTER
    if top_score < 0:
        print("⚠️ Warning: Low confidence score. Proceeding with caution...")
    
    # Format the chunks
    context_text = "\n\n".join([f"Source {i+1}:\n" + doc.page_content for i, doc in enumerate(context_chunks)])
    
    # 3. THE PROMPT (The 'Brain' of the AI)
    prompt = f"""
    You are a professional Project Management Assistant. 
    Use the following high-quality excerpts from the textbook to answer.
    
    Rules:
    1. Only use the provided context.
    2. If the info is missing, say "The textbook does not provide enough detail."
    3. Use a structured, professional tone.

    CONTEXT:
    {context_text}

    USER QUESTION:
    {query}

    FINAL ANSWER:
    """

    # 4. GENERATION
    print(f"🤖 DeepSeek-R1 is thinking using Precision Context...")
    start_time = time.time()
    response = ollama.generate(model='deepseek-r1:1.5b', prompt=prompt)
    
    print("-" * 30)
    print(f"📝 FINAL ANSWER:\n{response['response']}")
    print("-" * 30)
    print(f"⏱️ Generation Latency: {time.time() - start_time:.2f}s")
    print(f"🎯 Top Relevance Score: {top_score:.4f}")

In [23]:
# Testing your Master Function (v2)
test_query = "What is the definition of a project?"
ask_my_textbook_v2(test_query)

🔍 Deep Searching (Recall: 40, Precision: 5)...

--- 📊 RERANKER SCORECARD ---
Rank  | Score      | Content Snippet
----------------------------------------------------------------------
#1    |     7.3875 | 1.1 THE DEFINITION OF A “PROJECT” The PMI has deﬁ ned a proj...
#2    |     4.7355 | into tasks, which are, in turn, split into work packages tha...
#3    |     2.9917 | projects. Project management has emerged because the charact...
#4    |     1.8148 | spread to the nonproﬁ t sector. Weddings, scout-o-ramas, fun...
#5    |     1.7130 | sonnel, and any other matters will be reviewed to determine ...
#6    |     0.7299 | still projects, and if so, can project management methods be...
#7    |     0.6667 | project completion. Event An end state for one or more activ...
#8    |     0.5654 | They are complex, multidisciplinary, and have the same gener...
#9    |     0.5000 | which are the importance of an organized plan for reaching a...
#10   |     0.1693 | The project audit is a thorou

In [24]:
# Testing your Master Function (v2)
test_query = "What are the three primary goals of a project according to Meredith and Mantel?"
ask_my_textbook_v2(test_query)

🔍 Deep Searching (Recall: 40, Precision: 5)...

--- 📊 RERANKER SCORECARD ---
Rank  | Score      | Content Snippet
----------------------------------------------------------------------
#1    |    -1.5837 | and desires along with the “speciﬁ ed” performance, as state...
#2    |    -3.2904 | team members. In Chapter 4 we will discuss some issues invol...
#3    |    -3.3495 | project and are thus direct goals. Ancillary goals include i...
#4    |    -4.3411 | 4   CHAPTER 1 / PROJECTS IN CONTEMPORARY ORGANIZATIONS shown...
#5    |    -4.7886 | we must use considerable care when planning projects. The pr...
#6    |    -5.1128 | for the project’s existence, it is much easier to enlist and...
#7    |    -5.1880 | Thus, if some of the project’s objectives are not openly deb...
#8    |    -5.9779 | ment is the project’s environment, that is, those things or ...
#9    |    -6.8345 | Class Discussion Questions  13. What percentage of the total...
#10   |    -6.9250 | keeps projects from competing